# 95b — Stack waveform-QC-approved nodal-only shot groups (SAFE)

This notebook creates **stacked nodal shot gathers for acquisitions that were recorded only by the nodes and were not tied to a Geode stack record**.

It begins with the accepted-event manifest written by notebook **94c**:

`94c_nodal_only_candidates_accepted_for_future_stack.csv`

For each nodal-only source group, it:

1. validates the accepted membership and required waveform files;
2. reloads and preprocesses the long three-component gathers;
3. applies the event-level cross-correlation shifts determined in notebook 94c;
4. forms a linear mean stack independently for `DPZ`, `DPN`, and `DPE`;
5. writes MiniSEED, SEG-Y, and wiggle-plot products;
6. writes CSV provenance tables describing each stack, its members, output files, and any errors.

This notebook **does not modify the SQLite catalog**. It writes only to a new output directory. It does not alter notebooks 90–95 or their products.

### Important interpretation

The output represents one stacked nodal gather per accepted **nodal-only candidate source group**. Source positions are those assigned during notebook 94b and reviewed in notebook 94c. They are not independently surveyed Geode source coordinates and should remain identified as inferred or field-note-supported nodal-only positions.


## 1. Configuration

In [1]:
from pathlib import Path
import json
import traceback

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from obspy import read, Stream, Trace, UTCDateTime

PROJECT_ROOT = Path('/Volumes/tachyon/LBSSP_DATA')

QC_ROOT = PROJECT_ROOT / 'nodal_only_candidate_waveform_qc'
ACCEPTED_CSV = QC_ROOT / '94c_nodal_only_candidates_accepted_for_future_stack.csv'
GROUP_SUMMARY_CSV = QC_ROOT / '94c_nodal_only_group_waveform_qc_summary.csv'

OUT_ROOT = PROJECT_ROOT / 'nodal_only_stacked_shot_gathers'
STACK_ROOT = OUT_ROOT / 'stacks'
EXPORT_ROOT = OUT_ROOT / 'catalog_exports'
FIGURE_ROOT = OUT_ROOT / 'figures'

for directory in [OUT_ROOT, STACK_ROOT, EXPORT_ROOT, FIGURE_ROOT]:
    directory.mkdir(parents=True, exist_ok=True)

COMPONENTS_TO_STACK = ['Z', 'N', 'E']
MIN_EVENTS_PER_STACK = 2

# Match the preprocessing used by notebook 94c.
BANDPASS_FREQMIN_HZ = 5.0
BANDPASS_FREQMAX_HZ = 150.0

# Output time window relative to the event origin after applying the 94c shift.
STACK_TMIN_S = -0.05
STACK_TMAX_S = 1.20

# Require at least this many finite contributing events at a sample.
MIN_SAMPLE_CONTRIBUTORS = 1

WRITE_MSEED = True
WRITE_SEGY = True
WRITE_PNG = True
SHOW_PLOTS = False
OVERWRITE_OUTPUTS = True

# Set to a small integer for a test run, or None for all groups.
MAX_GROUPS = None

PLOT_TMIN_S = 0.0
PLOT_TMAX_S = 0.80
PLOT_CLIP_PERCENTILE = 99.0
PLOT_SCALE = 0.80

print('Accepted 94c manifest:', ACCEPTED_CSV)
print('Output root:', OUT_ROOT)
print('SQLite will not be opened or modified.')


Accepted 94c manifest: /Volumes/tachyon/LBSSP_DATA/nodal_only_candidate_waveform_qc/94c_nodal_only_candidates_accepted_for_future_stack.csv
Output root: /Volumes/tachyon/LBSSP_DATA/nodal_only_stacked_shot_gathers
SQLite will not be opened or modified.


## 2. Load and validate the notebook-94c accepted-event manifest

In [2]:
if not ACCEPTED_CSV.exists():
    raise FileNotFoundError(
        f'Missing accepted-event manifest: {ACCEPTED_CSV}\n'
        'Run notebook 94c first.'
    )

accepted = pd.read_csv(ACCEPTED_CSV, low_memory=False)

required_columns = {
    'group_key',
    'nodal_event_id',
    'event_time',
    'assigned_source_x_m',
    'resolved_gather_path',
    'accepted_for_future_stack',
    'xcorr_shift_s',
    'xcorr_corrcoef',
    'xcorr_n_traces',
    'xcorr_shift_mad_s',
}
missing_columns = sorted(required_columns - set(accepted.columns))
if missing_columns:
    raise RuntimeError(f'Accepted manifest is missing required columns: {missing_columns}')

accepted['event_time'] = pd.to_datetime(
    accepted['event_time'], utc=True, format='mixed', errors='coerce'
)
accepted['assigned_source_x_m'] = pd.to_numeric(
    accepted['assigned_source_x_m'], errors='coerce'
)
accepted['xcorr_shift_s'] = pd.to_numeric(
    accepted['xcorr_shift_s'], errors='coerce'
)
accepted['xcorr_corrcoef'] = pd.to_numeric(
    accepted['xcorr_corrcoef'], errors='coerce'
)
accepted['accepted_for_future_stack'] = (
    accepted['accepted_for_future_stack']
    .astype(str).str.lower().isin(['true', '1', 'yes'])
)

accepted = accepted.loc[accepted['accepted_for_future_stack']].copy()
accepted['gather_path_exists'] = accepted['resolved_gather_path'].map(
    lambda value: bool(value) and Path(str(value)).exists()
)

invalid = accepted.loc[
    accepted['event_time'].isna()
    | accepted['assigned_source_x_m'].isna()
    | accepted['xcorr_shift_s'].isna()
    | ~accepted['gather_path_exists']
].copy()

if len(invalid):
    display(
        invalid[
            ['group_key', 'nodal_event_id', 'event_time',
             'assigned_source_x_m', 'xcorr_shift_s',
             'resolved_gather_path', 'gather_path_exists']
        ].head(30)
    )
    raise RuntimeError(
        f'{len(invalid)} accepted rows have missing times, source positions, '
        'cross-correlation shifts, or waveform files.'
    )

group_counts = (
    accepted.groupby('group_key', as_index=False)
    .agg(
        assigned_source_x_m=('assigned_source_x_m', 'first'),
        n_accepted_members=('nodal_event_id', 'size'),
        first_event_utc=('event_time', 'min'),
        last_event_utc=('event_time', 'max'),
        median_corr=('xcorr_corrcoef', 'median'),
    )
    .sort_values(['first_event_utc', 'assigned_source_x_m'])
    .reset_index(drop=True)
)

group_counts = group_counts.loc[
    group_counts.n_accepted_members >= MIN_EVENTS_PER_STACK
].copy()

if MAX_GROUPS is not None:
    group_counts = group_counts.head(MAX_GROUPS).copy()

print('Accepted members:', len(accepted))
print('Groups eligible for stacking:', len(group_counts))
display(group_counts)


Accepted members: 832
Groups eligible for stacking: 44


,group_key,assigned_source_x_m,n_accepted_members,first_event_utc,last_event_utc,median_corr
0,MAY17_104M,104.0,7,2026-05-17 17:10:05+00:00,2026-05-17 17:10:44.160000+00:00,0.953017
1,MAY17_105M,105.0,20,2026-05-17 17:10:47.056000+00:00,2026-05-17 17:12:26.766000+00:00,0.977250
2,MAY17_106M,106.0,27,2026-05-17 17:12:28.822000+00:00,2026-05-17 17:14:29.408000+00:00,0.746471
3,MAY17_107M,107.0,21,2026-05-17 17:14:32.390000+00:00,2026-05-17 17:16:24.286000+00:00,0.716448
4,MAY17_108M,108.0,19,2026-05-17 17:16:27.150000+00:00,2026-05-17 17:17:48.118000+00:00,0.954830
5,MAY17_109M,109.0,21,2026-05-17 17:18:30.984000+00:00,2026-05-17 17:19:57.474000+00:00,0.959315
6,MAY17_110M,110.0,26,2026-05-17 17:20:00.216000+00:00,2026-05-17 17:22:14.508000+00:00,0.928301
7,MAY17_111M,111.0,16,2026-05-17 17:22:40.484000+00:00,2026-05-17 17:25:29.116000+00:00,0.944691
8,MAY17_112M,112.0,22,2026-05-17 17:26:08.560000+00:00,2026-05-17 17:28:07.938000+00:00,0.977234
9,MAY17_113M,113.0,19,2026-05-17 17:28:11.020000+00:00,2026-05-17 17:36:06.322000+00:00,0.941099


## 3. Waveform and geometry helpers

Receiver position is recovered from the position-coded station name, following notebook 94c. For example, station code `012300` is interpreted as 123.00 m when the station code is stored in hundredths of a metre.

The stack is a sample-by-sample arithmetic mean after applying the event-level shift from notebook 94c. A separate valid-sample count is retained internally so missing samples do not bias the mean.


In [3]:
def safe_name(value):
    text = str(value)
    for char in [' ', '/', '\\\\', ':', ';', ',', '(', ')', '[', ']']:
        text = text.replace(char, '_')
    return text


def station_x_m(trace):
    try:
        return int(str(trace.stats.station)) / 100.0
    except Exception:
        return np.nan


def preprocess_stream(stream):
    output = Stream()
    for original in stream:
        if not any(str(original.stats.channel).endswith(component)
                   for component in COMPONENTS_TO_STACK):
            continue

        trace = original.copy()
        trace.data = trace.data.astype(np.float64)
        trace.detrend('linear')
        trace.taper(max_percentage=0.02, type='hann')

        nyquist = 0.5 / trace.stats.delta
        if BANDPASS_FREQMAX_HZ < 0.95 * nyquist:
            trace.filter(
                'bandpass',
                freqmin=BANDPASS_FREQMIN_HZ,
                freqmax=BANDPASS_FREQMAX_HZ,
                corners=4,
                zerophase=True,
            )
        else:
            trace.filter(
                'highpass',
                freq=BANDPASS_FREQMIN_HZ,
                corners=4,
                zerophase=True,
            )

        trace.stats.receiver_x_m = station_x_m(trace)
        output += trace

    return output


def trace_key(trace):
    return str(trace.stats.station), str(trace.stats.channel)


def stream_by_key(stream):
    return {trace_key(trace): trace for trace in stream}


def build_stacked_stream(member_streams, member_origins, member_shifts, component):
    dt_values = [
        float(trace.stats.delta)
        for stream in member_streams.values()
        for trace in stream.select(channel=f'*{component}')
    ]
    if not dt_values:
        return Stream(), {}

    dt = float(np.median(dt_values))
    if any(abs(value - dt) > 1e-6 for value in dt_values):
        raise RuntimeError(f'Inconsistent sample intervals in component {component}')

    time_grid = np.arange(
        STACK_TMIN_S, STACK_TMAX_S + 0.5 * dt, dt, dtype=float
    )

    keys = sorted({
        trace_key(trace)
        for stream in member_streams.values()
        for trace in stream.select(channel=f'*{component}')
        if np.isfinite(float(getattr(trace.stats, 'receiver_x_m', np.nan)))
    })

    output = Stream()
    contribution_summary = {}

    for key in keys:
        arrays = []
        template = None
        receiver_x = np.nan
        contributing_event_ids = []

        for event_id, stream in member_streams.items():
            trace = stream_by_key(stream).get(key)
            if trace is None:
                continue

            origin = member_origins[event_id]
            shift = float(member_shifts[event_id])

            trace_start = trace.stats.starttime - origin
            trace_times = (
                trace_start
                + np.arange(trace.stats.npts, dtype=float) * trace.stats.delta
            )

            # Same sign convention as notebook 95:
            # evaluate each candidate at t - shift to align it to the reference.
            values = np.interp(
                time_grid - shift,
                trace_times,
                trace.data.astype(float),
                left=np.nan,
                right=np.nan,
            )

            arrays.append(values)
            contributing_event_ids.append(event_id)

            if template is None:
                template = trace
                receiver_x = float(trace.stats.receiver_x_m)

        if not arrays or template is None:
            continue

        array = np.vstack(arrays)
        valid_counts = np.sum(np.isfinite(array), axis=0)
        stacked = np.divide(
            np.nansum(array, axis=0),
            valid_counts,
            out=np.zeros(array.shape[1], dtype=float),
            where=valid_counts >= MIN_SAMPLE_CONTRIBUTORS,
        )

        stats = template.stats.copy()
        stats.starttime = UTCDateTime(0) + STACK_TMIN_S
        stats.delta = dt
        stats.npts = len(stacked)
        stats.receiver_x_m = receiver_x
        stats.processing = list(getattr(stats, 'processing', [])) + [
            f'linear mean stack of {len(arrays)} nodal-only events by notebook 95b',
            'event shifts imported from notebook 94c',
        ]
        stats.pop('mseed', None)

        output += Trace(data=stacked.astype(np.float32), header=stats)
        contribution_summary[key] = {
            'receiver_x_m': receiver_x,
            'n_contributing_events': len(arrays),
            'contributing_event_ids': contributing_event_ids,
            'minimum_valid_samples': int(valid_counts.min()),
            'median_valid_samples': float(np.median(valid_counts)),
            'maximum_valid_samples': int(valid_counts.max()),
        }

    return output, contribution_summary


def plot_wiggle_stream(stream, title, source_x_m=None, output_path=None):
    traces = [
        trace for trace in stream
        if np.isfinite(float(getattr(trace.stats, 'receiver_x_m', np.nan)))
    ]
    if not traces:
        raise RuntimeError(f'No traces with receiver_x_m for plot: {title}')

    traces = sorted(traces, key=lambda trace: float(trace.stats.receiver_x_m))
    dt = float(np.median([trace.stats.delta for trace in traces]))
    time_grid = np.arange(PLOT_TMIN_S, PLOT_TMAX_S + 0.5 * dt, dt)

    receiver_positions = []
    waveform_rows = []
    for trace in traces:
        trace_times = (
            trace.stats.starttime - UTCDateTime(0)
            + np.arange(trace.stats.npts, dtype=float) * trace.stats.delta
        )
        values = np.interp(
            time_grid, trace_times, trace.data.astype(float),
            left=np.nan, right=np.nan,
        )
        if np.isfinite(values).sum() < 5:
            continue
        values = values - np.nanmedian(values)
        receiver_positions.append(float(trace.stats.receiver_x_m))
        waveform_rows.append(values)

    if not waveform_rows:
        raise RuntimeError(f'No valid waveform arrays for plot: {title}')

    positions = np.asarray(receiver_positions, dtype=float)
    waveform_array = np.vstack(waveform_rows)

    clip = np.nanpercentile(np.abs(waveform_array), PLOT_CLIP_PERCENTILE)
    if not np.isfinite(clip) or clip <= 0:
        clip = np.nanmax(np.abs(waveform_array))
    if not np.isfinite(clip) or clip <= 0:
        clip = 1.0

    unique_positions = np.sort(np.unique(positions))
    spacing = (
        np.nanmedian(np.diff(unique_positions))
        if len(unique_positions) > 1 else 1.0
    )
    if not np.isfinite(spacing) or spacing <= 0:
        spacing = 1.0

    figure, axis = plt.subplots(figsize=(12, 6))
    for position, values in zip(positions, waveform_array):
        scaled = np.clip(values / clip, -1, 1) * spacing * PLOT_SCALE
        axis.plot(position + scaled, time_grid, linewidth=0.7)
        axis.fill_betweenx(
            time_grid, position, position + np.maximum(scaled, 0), alpha=0.25
        )

    if source_x_m is not None and np.isfinite(source_x_m):
        axis.axvline(
            float(source_x_m), linestyle='--', linewidth=1.2,
            label=f'assigned source x={source_x_m:.1f} m',
        )
        axis.legend(loc='best')

    axis.invert_yaxis()
    axis.set_xlabel('Receiver x (m)')
    axis.set_ylabel('Time relative to aligned stack origin (s)')
    axis.set_title(title)
    axis.grid(True, alpha=0.25)
    figure.tight_layout()

    if output_path is not None:
        output_path = Path(output_path)
        output_path.parent.mkdir(parents=True, exist_ok=True)
        figure.savefig(output_path, dpi=180, bbox_inches='tight')

    if SHOW_PLOTS:
        plt.show()
    else:
        plt.close(figure)


def write_segy(stream, output_path, source_x_m):
    from obspy.io.segy.segy import SEGYTraceHeader
    from obspy.core import AttribDict

    segy_stream = stream.copy()
    for sequence, trace in enumerate(segy_stream, start=1):
        if not hasattr(trace.stats, 'segy'):
            trace.stats.segy = AttribDict()
        trace.stats.segy.trace_header = SEGYTraceHeader()
        header = trace.stats.segy.trace_header

        receiver_x = float(getattr(trace.stats, 'receiver_x_m', np.nan))
        source_x = float(source_x_m)

        # Coordinates are written in centimetres using scalar -100.
        header.trace_sequence_number_within_line = sequence
        header.trace_sequence_number_within_segy_file = sequence
        header.coordinate_scalar = -100
        header.source_coordinate_x = int(round(source_x * 100))

        if np.isfinite(receiver_x):
            header.group_coordinate_x = int(round(receiver_x * 100))
            header.distance_from_center_of_the_source_point_to_the_center_of_the_receiver_group = int(
                round((receiver_x - source_x) * 100)
            )

    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    segy_stream.write(str(output_path), format='SEGY', data_encoding=5)


## 4. Build the nodal-only stacked gathers

In [4]:
stack_rows = []
member_rows = []
file_rows = []
trace_contribution_rows = []
error_rows = []

for group_number, group_row in group_counts.iterrows():
    group_key = str(group_row['group_key'])
    print(f'[{len(stack_rows) + len(error_rows) + 1}/{len(group_counts)}] {group_key}')

    members = (
        accepted.loc[accepted.group_key.astype(str).eq(group_key)]
        .sort_values('event_time')
        .reset_index(drop=True)
    )

    source_positions = members.assigned_source_x_m.dropna().unique()
    if len(source_positions) != 1:
        error_rows.append({
            'group_key': group_key,
            'stage': 'validate_group',
            'error': f'Expected one assigned source position, found {source_positions.tolist()}',
            'traceback': None,
        })
        continue

    source_x_m = float(source_positions[0])
    stack_id = f'NODALONLYSTACK_{safe_name(group_key)}_x{source_x_m:06.1f}m'
    stack_directory = STACK_ROOT / safe_name(group_key)
    stack_directory.mkdir(parents=True, exist_ok=True)

    try:
        member_streams = {}
        member_origins = {}
        member_shifts = {}

        for _, member in members.iterrows():
            event_id = str(member['nodal_event_id'])
            stream = preprocess_stream(read(str(member['resolved_gather_path'])))
            if not stream:
                raise RuntimeError(f'Empty processed stream for {event_id}')

            member_streams[event_id] = stream
            member_origins[event_id] = UTCDateTime(
                member['event_time'].to_pydatetime()
            )
            member_shifts[event_id] = float(member['xcorr_shift_s'])

            member_record = {
                'stack_id': stack_id,
                'group_key': group_key,
                'assigned_source_x_m': source_x_m,
                'nodal_event_id': event_id,
                'event_time_utc': member['event_time'].isoformat(),
                'corrected_event_time_utc': (
                    member.get('corrected_event_time_utc')
                    if pd.notna(member.get('corrected_event_time_utc')) else None
                ),
                'xcorr_shift_s': float(member['xcorr_shift_s']),
                'xcorr_corrcoef': float(member['xcorr_corrcoef']),
                'xcorr_n_traces': int(member['xcorr_n_traces']),
                'xcorr_shift_mad_s': float(member['xcorr_shift_mad_s']),
                'is_reference_event': bool(member.get('is_reference_event', False)),
                'waveform_qc_status': member.get('waveform_qc_status'),
                'resolved_gather_path': str(member['resolved_gather_path']),
                'included_in_stack': True,
            }
            for optional_column in [
                'expected_blows', 'group_origin', 'nodal_timewindow_label',
                'assignment_method', 'recovery_score',
            ]:
                if optional_column in members.columns:
                    member_record[optional_column] = member.get(optional_column)
            member_rows.append(member_record)

        component_outputs = 0
        component_trace_counts = {}

        for component in COMPONENTS_TO_STACK:
            stacked_stream, contribution_summary = build_stacked_stream(
                member_streams, member_origins, member_shifts, component
            )
            if not stacked_stream:
                continue

            component_outputs += 1
            component_trace_counts[component] = len(stacked_stream)
            component_directory = stack_directory / component
            component_directory.mkdir(parents=True, exist_ok=True)
            basename = (
                f'{stack_id}_DP{component}_{len(members):03d}ev'
            )

            if WRITE_MSEED:
                mseed_path = component_directory / f'{basename}.mseed'
                if OVERWRITE_OUTPUTS or not mseed_path.exists():
                    stacked_stream.write(
                        str(mseed_path), format='MSEED', encoding='FLOAT32'
                    )
                file_rows.append({
                    'stack_id': stack_id,
                    'group_key': group_key,
                    'component': component,
                    'file_type': 'mseed',
                    'file_path': str(mseed_path),
                    'n_traces': len(stacked_stream),
                    'n_stack_members': len(members),
                })

            if WRITE_SEGY:
                segy_path = component_directory / f'{basename}.sgy'
                if OVERWRITE_OUTPUTS or not segy_path.exists():
                    write_segy(stacked_stream, segy_path, source_x_m)
                file_rows.append({
                    'stack_id': stack_id,
                    'group_key': group_key,
                    'component': component,
                    'file_type': 'segy',
                    'file_path': str(segy_path),
                    'n_traces': len(stacked_stream),
                    'n_stack_members': len(members),
                })

            if WRITE_PNG:
                png_path = component_directory / f'{basename}_wiggle.png'
                if OVERWRITE_OUTPUTS or not png_path.exists():
                    plot_wiggle_stream(
                        stacked_stream,
                        title=(
                            f'{group_key}: nodal-only stacked gather\n'
                            f'assigned source x={source_x_m:.1f} m; '
                            f'{len(members)} accepted blows; DP{component}'
                        ),
                        source_x_m=source_x_m,
                        output_path=png_path,
                    )
                file_rows.append({
                    'stack_id': stack_id,
                    'group_key': group_key,
                    'component': component,
                    'file_type': 'png_wiggle',
                    'file_path': str(png_path),
                    'n_traces': len(stacked_stream),
                    'n_stack_members': len(members),
                })

            for (station, channel), detail in contribution_summary.items():
                trace_contribution_rows.append({
                    'stack_id': stack_id,
                    'group_key': group_key,
                    'component': component,
                    'station': station,
                    'channel': channel,
                    'receiver_x_m': detail['receiver_x_m'],
                    'n_contributing_events': detail['n_contributing_events'],
                    'minimum_valid_samples': detail['minimum_valid_samples'],
                    'median_valid_samples': detail['median_valid_samples'],
                    'maximum_valid_samples': detail['maximum_valid_samples'],
                    'contributing_event_ids_json': json.dumps(
                        detail['contributing_event_ids']
                    ),
                })

        if component_outputs == 0:
            raise RuntimeError('No component produced a non-empty stacked stream')

        expected_values = (
            pd.to_numeric(members.get('expected_blows'), errors='coerce')
            if 'expected_blows' in members.columns else pd.Series(dtype=float)
        )

        stack_rows.append({
            'stack_id': stack_id,
            'group_key': group_key,
            'stack_basis': 'nodal_only_inferred_group',
            'assigned_source_x_m': source_x_m,
            'source_position_status': 'assigned_in_94b_reviewed_in_94c',
            'n_accepted_members': len(members),
            'expected_blows': (
                float(expected_values.dropna().iloc[0])
                if len(expected_values.dropna()) else np.nan
            ),
            'reference_nodal_event_id': (
                members.loc[
                    members.get('is_reference_event', False).astype(bool),
                    'nodal_event_id',
                ].iloc[0]
                if 'is_reference_event' in members.columns
                and members.get('is_reference_event', False).astype(bool).any()
                else None
            ),
            'median_xcorr_shift_s': float(np.nanmedian(members.xcorr_shift_s)),
            'median_xcorr_corrcoef': float(np.nanmedian(members.xcorr_corrcoef)),
            'minimum_xcorr_corrcoef': float(np.nanmin(members.xcorr_corrcoef)),
            'first_member_utc': members.event_time.min().isoformat(),
            'last_member_utc': members.event_time.max().isoformat(),
            'components_written': ','.join(
                component for component in COMPONENTS_TO_STACK
                if component in component_trace_counts
            ),
            'component_trace_counts_json': json.dumps(component_trace_counts),
            'output_directory': str(stack_directory),
            'status': 'stack_created',
        })

    except Exception as exc:
        error_rows.append({
            'group_key': group_key,
            'stage': 'stack_group',
            'error': repr(exc),
            'traceback': traceback.format_exc(),
        })
        print('  ERROR:', repr(exc))

nodal_only_stacks = pd.DataFrame(stack_rows)
nodal_only_stack_members = pd.DataFrame(member_rows)
nodal_only_stack_files = pd.DataFrame(file_rows)
nodal_only_stack_trace_contributions = pd.DataFrame(trace_contribution_rows)
nodal_only_stack_errors = pd.DataFrame(
    error_rows, columns=['group_key', 'stage', 'error', 'traceback']
)

print('Stacks created:', len(nodal_only_stacks))
print('Stack members:', len(nodal_only_stack_members))
print('Output files indexed:', len(nodal_only_stack_files))
print('Errors:', len(nodal_only_stack_errors))


[1/44] MAY17_104M
[2/44] MAY17_105M
[3/44] MAY17_106M
[4/44] MAY17_107M
[5/44] MAY17_108M
[6/44] MAY17_109M
[7/44] MAY17_110M
[8/44] MAY17_111M
[9/44] MAY17_112M
[10/44] MAY17_113M
[11/44] MAY17_114M
[12/44] MAY17_115M
[13/44] MAY17_116M
[14/44] MAY17_117M
[15/44] MAY17_118M
[16/44] MAY17_119M
[17/44] MAY17_120M
[18/44] MAY17_121M
[19/44] MAY17_122M
[20/44] MAY17_123M
[21/44] MAY17_124M
[22/44] MAY17_125M
[23/44] MAY17_126M
[24/44] MAY17_127M
[25/44] MAY17_128M
[26/44] MAY17_129M
[27/44] MAY17_130M
[28/44] MAY17_131M
[29/44] MAY17_132M
[30/44] MAY17_133M
[31/44] MAY17_134M
[32/44] MAY17_135M
[33/44] MAY17_136M
[34/44] MAY17_137M
[35/44] MAY17_138M
[36/44] MAY17_139M
[37/44] MAY17_140M
[38/44] MAY19_036M
[39/44] MAY19_SINKHOLE_110_114M
[40/44] MAY19_122M
[41/44] MAY19_130M
[42/44] MAY19_216M
[43/44] MAY19_010M
[44/44] MAY19_290M
Stacks created: 44
Stack members: 832
Output files indexed: 396
Errors: 0


## 5. Export stack provenance and QC summaries

In [6]:
def rounded_export(frame):
    output = frame.copy()
    for column in [
        'assigned_source_x_m', 'receiver_x_m',
        'xcorr_shift_s', 'xcorr_shift_mad_s',
        'median_xcorr_shift_s',
    ]:
        if column in output.columns:
            output[column] = pd.to_numeric(
                output[column], errors='coerce'
            ).round(3)
    for column in [
        'xcorr_corrcoef', 'median_xcorr_corrcoef',
        'minimum_xcorr_corrcoef',
    ]:
        if column in output.columns:
            output[column] = pd.to_numeric(
                output[column], errors='coerce'
            ).round(4)
    return output


rounded_export(nodal_only_stacks).to_csv(
    EXPORT_ROOT / 'nodal_only_stacks.csv', index=False
)
rounded_export(nodal_only_stack_members).to_csv(
    EXPORT_ROOT / 'nodal_only_stack_members.csv', index=False
)
nodal_only_stack_files.to_csv(
    EXPORT_ROOT / 'nodal_only_stack_files.csv', index=False
)
rounded_export(nodal_only_stack_trace_contributions).to_csv(
    EXPORT_ROOT / 'nodal_only_stack_trace_contributions.csv', index=False
)
nodal_only_stack_errors.to_csv(
    EXPORT_ROOT / 'nodal_only_stack_processing_errors.csv', index=False
)

if len(nodal_only_stacks):

    summary = pd.DataFrame([{
        "n_stacks": len(nodal_only_stacks),
        "total_stack_members": nodal_only_stacks["n_accepted_members"].sum(),
        "median_members_per_stack": nodal_only_stacks["n_accepted_members"].median(),
        "minimum_members_per_stack": nodal_only_stacks["n_accepted_members"].min(),
        "maximum_members_per_stack": nodal_only_stacks["n_accepted_members"].max(),
        "median_stack_correlation": nodal_only_stacks["median_xcorr_corrcoef"].median(),
    }])

    display(summary)
    display(summary)

    display(
        nodal_only_stacks[
            [
                'group_key', 'assigned_source_x_m',
                'n_accepted_members', 'expected_blows',
                'median_xcorr_corrcoef', 'components_written',
                'status',
            ]
        ].sort_values('assigned_source_x_m')
    )

if len(nodal_only_stack_files):
    display(
        nodal_only_stack_files
        .groupby(['component', 'file_type'], dropna=False)
        .size()
        .reset_index(name='n_files')
    )

if len(nodal_only_stack_errors):
    print('Processing errors:')
    display(nodal_only_stack_errors[['group_key', 'stage', 'error']])

print('Stack products:', STACK_ROOT)
print('CSV provenance:', EXPORT_ROOT)


,n_stacks,total_stack_members,median_members_per_stack,minimum_members_per_stack,maximum_members_per_stack,median_stack_correlation
0,44,832,18.0,6,62,0.931842


,n_stacks,total_stack_members,median_members_per_stack,minimum_members_per_stack,maximum_members_per_stack,median_stack_correlation
0,44,832,18.0,6,62,0.931842


,group_key,assigned_source_x_m,n_accepted_members,expected_blows,median_xcorr_corrcoef,components_written,status
42,MAY19_010M,10.0,43,60.0,0.968003,"Z,N,E",stack_created
37,MAY19_036M,36.0,22,22.0,0.978909,"Z,N,E",stack_created
0,MAY17_104M,104.0,7,20.0,0.953017,"Z,N,E",stack_created
1,MAY17_105M,105.0,20,20.0,0.977250,"Z,N,E",stack_created
2,MAY17_106M,106.0,27,20.0,0.746471,"Z,N,E",stack_created
3,MAY17_107M,107.0,21,20.0,0.716448,"Z,N,E",stack_created
4,MAY17_108M,108.0,19,20.0,0.954830,"Z,N,E",stack_created
5,MAY17_109M,109.0,21,20.0,0.959315,"Z,N,E",stack_created
6,MAY17_110M,110.0,26,20.0,0.928301,"Z,N,E",stack_created
7,MAY17_111M,111.0,16,20.0,0.944691,"Z,N,E",stack_created


,component,file_type,n_files
0,E,mseed,44
1,E,png_wiggle,44
2,E,segy,44
3,N,mseed,44
4,N,png_wiggle,44
5,N,segy,44
6,Z,mseed,44
7,Z,png_wiggle,44
8,Z,segy,44


Stack products: /Volumes/tachyon/LBSSP_DATA/nodal_only_stacked_shot_gathers/stacks
CSV provenance: /Volumes/tachyon/LBSSP_DATA/nodal_only_stacked_shot_gathers/catalog_exports


## 6. Final integrity check

A successful run should produce one row in `nodal_only_stacks.csv` for every eligible 94c group, along with component-specific MiniSEED, SEG-Y, and PNG files. The integrity check below verifies that every indexed output exists and that every successful stack has at least one component.


In [7]:
if len(nodal_only_stack_files):
    nodal_only_stack_files['file_exists'] = nodal_only_stack_files.file_path.map(
        lambda value: Path(str(value)).exists()
    )
    missing_files = nodal_only_stack_files.loc[
        ~nodal_only_stack_files.file_exists
    ]
else:
    missing_files = pd.DataFrame()

expected_groups = set(group_counts.group_key.astype(str))
successful_groups = set(nodal_only_stacks.group_key.astype(str))
failed_or_missing_groups = sorted(expected_groups - successful_groups)

print('Eligible groups:', len(expected_groups))
print('Successful groups:', len(successful_groups))
print('Failed or missing groups:', len(failed_or_missing_groups))
print('Indexed files missing on disk:', len(missing_files))

if failed_or_missing_groups:
    print('Groups without completed stacks:')
    print(failed_or_missing_groups)

if len(missing_files):
    display(missing_files)

if not failed_or_missing_groups and len(missing_files) == 0:
    print('PASS: all eligible nodal-only groups produced indexed stack products.')
else:
    print('REVIEW: inspect processing errors and missing-file diagnostics.')

print()
print('These are nodal-only stacked shot gathers.')
print(
    'They complement the Geode-associated nodal stacks from notebook 95, '
    'but retain separate provenance because their source groups were inferred '
    'or field-note-supported rather than keyed to Geode stack records.'
)


Eligible groups: 44
Successful groups: 44
Failed or missing groups: 0
Indexed files missing on disk: 0
PASS: all eligible nodal-only groups produced indexed stack products.

These are nodal-only stacked shot gathers.
They complement the Geode-associated nodal stacks from notebook 95, but retain separate provenance because their source groups were inferred or field-note-supported rather than keyed to Geode stack records.
